# Batch Scoring — Content Completion Prediction

Scores every viewing session; writes predictions + per-genre engagement summary.

**Reads:** `gold_ml_features` + saved model  **Writes:** `gold_ml_predictions`, `gold_ml_summary`

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, when, avg, count, isnan, udf,
    sum as spark_sum, round as spark_round
)
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassificationModel

spark = SparkSession.builder.getOrCreate()
model = RandomForestClassificationModel.load('Files/models/completion_rf')
df = spark.read.table('gold_ml_features')
print(f'Scoring {df.count():,} feature rows')

In [ ]:
for c, dtype in df.dtypes:
    if dtype in ('double', 'float'):
        df = df.withColumn(c, when(col(c).isNull() | isnan(col(c)), lit(0.0)).otherwise(col(c)))
    elif dtype in ('int', 'bigint', 'long'):
        df = df.withColumn(c, when(col(c).isNull(), lit(0)).otherwise(col(c)))

numeric_features = [
    'duration_mins', 'release_year', 'monthly_fee',
    'view_hour', 'view_dow', 'is_weekend',
]
cat_cols = ['genre', 'content_type', 'production_cost_bucket', 'language',
            'plan_type', 'region', 'age_group', 'device_type']
indexed_df = df
for c in cat_cols:
    idx_col = f'{c}_idx'
    indexer = StringIndexer(inputCol=c, outputCol=idx_col, handleInvalid='keep')
    indexed_df = indexer.fit(indexed_df).transform(indexed_df)

all_features = numeric_features + [f'{c}_idx' for c in cat_cols]
assembler = VectorAssembler(inputCols=all_features, outputCol='features', handleInvalid='keep')
model_df = assembler.transform(indexed_df)

In [ ]:
scored = model.transform(model_df)
extract_prob = udf(lambda v: float(v[1]) if v is not None and len(v) > 1 else 0.0, DoubleType())

predictions = (
    scored
    .withColumn('complete_probability', spark_round(extract_prob(col('probability')), 4))
    .withColumn('predicted_complete', col('prediction').cast('int'))
    .withColumn('engagement_level',
        when(col('complete_probability') > 0.8, 'very_high')
        .when(col('complete_probability') > 0.6, 'high')
        .when(col('complete_probability') > 0.4, 'medium')
        .otherwise('low'))
    .withColumn('scored_at', current_timestamp())
    .select(
        'view_id', 'subscriber_id', 'content_id', 'genre', 'content_type', 'device_type',
        'plan_type', 'duration_mins',
        'had_complete', 'predicted_complete', 'complete_probability', 'engagement_level',
        'scored_at')
)
predictions.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_predictions')
print(f'Predictions written: {predictions.count():,} rows')
predictions.groupBy('engagement_level').count().orderBy('count', ascending=False).show()

In [ ]:
# Per-genre engagement summary
summary = (
    predictions
    .groupBy('genre')
    .agg(
        count('*').alias('total_views'),
        spark_sum('predicted_complete').alias('predicted_complete_count'),
        spark_sum('had_complete').alias('actual_complete_count'),
        spark_round(avg('complete_probability'), 4).alias('avg_complete_probability'),
        spark_round(avg('duration_mins'), 1).alias('avg_duration_mins'),
    )
    .withColumn('completion_rate', spark_round(col('predicted_complete_count') / col('total_views') * 100, 1))
    .withColumn('overall_engagement',
        when(col('avg_complete_probability') > 0.6, 'high')
        .when(col('avg_complete_probability') > 0.4, 'medium')
        .otherwise('low'))
    .withColumn('summary_timestamp', current_timestamp())
    .orderBy(col('avg_complete_probability').desc())
)
summary.write.mode('overwrite').option('overwriteSchema', 'true').format('delta').saveAsTable('gold_ml_summary')
print(f'Genre engagement summary: {summary.count()} rows')
summary.show(15, truncate=False)

In [ ]:
spark.sql('OPTIMIZE gold_ml_predictions')
spark.sql('OPTIMIZE gold_ml_summary')
print('All Gold ML tables optimized')